# Block 1: Basic Elements in Neural Networks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatge-upc/aa2/blob/main/notebooks/aa2_1_2_backpropagation.ipynb)

## 1.2: Backpropagation

This notebook introduces the fundamental concepts of backpropagation using PyTorch:

*   **Computational Graphs**: Following the intermediate steps of a forward pass.
*   **Automatic Differentiation**: Computing gradients with `requires_grad` and `.backward()`.
*   **The Chain Rule**: Inspecting how gradients propagate through intermediate tensors with `retain_grad()`.
*   **Shared Parameters**: Adding gradient contributions from multiple examples and understanding gradient accumulation.
*   **Sum vs. Mean Loss**: Exploring how loss aggregation affects the gradients.

We want to learn to **multiply by 2**, using the examples $(x_1, y_1)=(2,4)$ and $(x_2, y_2)=(3,6)$. Our model is $\hat{y}=xw+b$, with squared error $SE(\hat{y},y)=(\hat{y}-y)^2$.

We choose $w=1$ and $b=1$ as initial values. Here we compute gradients; updating the parameters is a separate step.

### Forward Pass: Follow the Intermediate Steps

For the first example, compute $a=xw$, then $\hat{y}=a+b$, then the squared error. PyTorch builds the computational graph as these operations run.

In [ ]:
import torch

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
x = torch.tensor(2.0)
y = torch.tensor(4.0)

a = x * w
y_hat = a + b
loss = (y_hat - y)**2

for name, tensor in [("w", w), ("b", b), ("a", a), ("y_hat", y_hat), ("loss", loss)]:
    print(f"{name}: {tensor!r}")

The intermediate values are $a=2$, $\hat{y}=3$, and $L=1$ (loss). Their `grad_fn` identifies the recorded backward operation: a hint that the computational graph has been built. The parameters `w` and `b` are **leaf tensors**, created directly with `requires_grad=True`; they have no `grad_fn`.

### Backward Pass: Inspect the Gradients

`loss.backward()` applies the chain rule. Intermediate gradients are not stored by default; `retain_grad()` lets us inspect them.

In [ ]:
a.retain_grad()
y_hat.retain_grad()

loss.backward()

print(f"dL/dy_hat: {y_hat.grad}")  # -2
print(f"dL/da: {a.grad}")  # -2
print(f"dL/dw: {w.grad}")  # -4
print(f"dL/db: {b.grad}")  # -2

**Question:** Why does the gradient become -4 on the way to `w`, but stay -2 on the way to `b`?

### Shared Parameters: Two Paths to the Loss

Both examples use the **same** `w` and `b`. For $L=L_1+L_2$, the scalar-valued multivariate chain rule adds the contributions through both branches:

$$
\frac{\partial L}{\partial w}
= \frac{\partial L}{\partial \hat{y}_1}\frac{\partial \hat{y}_1}{\partial w}
+ \frac{\partial L}{\partial \hat{y}_2}\frac{\partial \hat{y}_2}{\partial w}
= (-2)\cdot 2 + (-4)\cdot 3 = -16.
$$

We build a new graph and clear the previous gradients, since `.backward()` adds to the parameters' `.grad` fields.

In [ ]:
# Reset gradients: backward() accumulates them by default.
w.grad = None
b.grad = None

x1, y1 = torch.tensor(2.0), torch.tensor(4.0)
x2, y2 = torch.tensor(3.0), torch.tensor(6.0)

a1 = x1 * w
y_hat1 = a1 + b
loss1 = (y_hat1 - y1)**2

a2 = x2 * w
y_hat2 = a2 + b
loss2 = (y_hat2 - y2)**2

loss = loss1 + loss2
loss.backward()

print(f"Total loss: {loss.item()}")  # 1 + 4 = 5
print(f"dL/dw: {w.grad}")  # -4 + (-12) = -16
print(f"dL/db: {b.grad}")  # -2 + (-4) = -6

**Question:** If we used the mean loss $(L_1+L_2)/2$ instead of the sum, how would the gradients change?